In [6]:
%pip install -q python-dotenv openai openpyxl langchain-community langchain-text-splitters sentence-transformers faiss-cpu pypdf cryptography ragas

Note: you may need to restart the kernel to use updated packages.


In [7]:
# imports
import json as _json
import os
import pickle
import re
import time
from typing import Dict, List

import pandas as pd
from dotenv import load_dotenv
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from openai import OpenAI
from ragas.dataset_schema import SingleTurnSample
from ragas.llms import llm_factory
from ragas.metrics import Faithfulness  # was: ragas.metrics.collections
from tqdm.auto import tqdm

/var/folders/4p/3v7c85y55qdc7jt3wv838n700000gp/T/ipykernel_3823/36588356.py:18: DeprecationWarning: Importing Faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import Faithfulness
  from ragas.metrics import Faithfulness  # was: ragas.metrics.collections


In [8]:
# constants

COLAB = False

if COLAB:
    from google.colab import userdata

    NEBIUS_API_KEY = userdata.get("NEBIUS_API_KEY")
else:
    load_dotenv()
    NEBIUS_API_KEY = os.environ.get("NEBIUS_API_KEY")

NEBIUS_BASE_URL = "https://api.studio.nebius.ai/v1/"
PDFS_PATH = "https://raw.githubusercontent.com/patronus-ai/financebench/main/pdfs"
RAG_MODEL = "meta-llama/Llama-3.3-70B-Instruct"
EMBED_MODEL = "BAAI/bge-small-en-v1.5"
JUDGE_MODEL = "deepseek-ai/DeepSeek-V3.2"
VECTORSTORE_DIR = "financebench_rag_faiss"

client = OpenAI(api_key=NEBIUS_API_KEY, base_url=NEBIUS_BASE_URL)
print("Client ready ✓")

Client ready ✓


In [9]:
# dataset
def get_dataset():
    df = pd.read_json(
        "hf://datasets/PatronusAI/financebench/financebench_merged.jsonl",
        lines=True,
    )
    # Drop the metrics-generated questions
    df = df[df["question_type"] != "metrics-generated"]

    # For each row, replace the "doc_link" value with the url from "https://github.com/patronus-ai/financebench/tree/main/pdfs" such that the url is: "https://github.com/patronus-ai/financebench/tree/main/pdfs/{{doc_name}}.pdf"
    df["doc_link"] = df["doc_name"].apply(lambda x: f"{PDFS_PATH}/{x}.pdf")
    df = df.sort_values(by="financebench_id", ascending=True).reset_index(drop=True)

    return df


df = get_dataset()
print("Columns:", df.columns.tolist())
print("Unique question types:", df.question_type.unique().tolist())
df.head(2)

Columns: ['financebench_id', 'company', 'doc_name', 'question_type', 'question_reasoning', 'domain_question_num', 'question', 'answer', 'justification', 'dataset_subset_label', 'evidence', 'gics_sector', 'doc_type', 'doc_period', 'doc_link']
Unique question types: ['domain-relevant', 'novel-generated']


,financebench_id,company,doc_name,question_type,question_reasoning,domain_question_num,question,answer,justification,dataset_subset_label,evidence,gics_sector,doc_type,doc_period,doc_link
0,financebench_id_00005,Corning,CORNING_2022_10K,domain-relevant,Numerical reasoning OR Logical reasoning,dg24,Does Corning have positive working capital bas...,Yes. Corning had a positive working capital am...,"Trade accounts receivable, net of doubtful acc...",OPEN_SOURCE,[{'evidence_text': 'Consolidated Balance Sheet...,Information Technology,10k,2022,https://raw.githubusercontent.com/patronus-ai/...
1,financebench_id_00070,American Water Works,AMERICANWATERWORKS_2022_10K,domain-relevant,Numerical reasoning OR Logical reasoning,dg24,Does American Water Works have positive workin...,"No, American Water Works had negative working ...",Accounts receivable+Income tax receivable+Unbi...,OPEN_SOURCE,[{'evidence_text': 'American Water Works Compa...,Utilities,10k,2022,https://raw.githubusercontent.com/patronus-ai/...


---
## Task 1 - Naive Generation

In [10]:
# answer the first 5 questions of each question_type - 5 domain-relevant, 5 novel-generated

TASK_1_FILENAME = "assignment2_naive_generation.xlsx"

if os.path.exists(TASK_1_FILENAME):
    print("Loading existing results...")
    answers_df = pd.read_excel(TASK_1_FILENAME)
else:
    # Select the first 5 questions for each question_type
    questions_domain = df[df["question_type"] == "domain-relevant"].head(5)
    questions_novel = df[df["question_type"] == "novel-generated"].head(5)
    selected_questions = pd.concat([questions_domain, questions_novel]).reset_index(
        drop=True
    )

    answers = []

    for idx, row in selected_questions.iterrows():
        prompt = f"""
Answer the question in 2-4 sentences.
If you don't know the answer, say "I don't know".
QUESTION: {row["question"]}
ANSWER:
"""
        response = client.chat.completions.create(
            model=RAG_MODEL,
            messages=[{"role": "user", "content": prompt}],
            temperature=0.1,
            max_tokens=200,
        )
        naive_answer = response.choices[0].message.content.strip()
        result = pd.Series(
            {
                "financebench_id": row["financebench_id"],
                "question_type": row["question_type"],
                "question": row["question"],
                "naive_answer": naive_answer,
                "ground_truth": row["answer"],
                "verdict": "",  # correct/partially correct/wrong/refused
            }
        )
        answers.append(result)
        time.sleep(1.5)  # rate limiting

    answers_df = pd.DataFrame(answers)

    # save results before setting the verdict
    answers_df.to_excel(TASK_1_FILENAME, index=False)

Loading existing results...


In [11]:
# set verdict
VERDICT_MAP = {
    1: "correct",
    2: "partially correct",
    3: "wrong",
    4: "refused",
}
verdicts_dict = {
    "financebench_id_00005": VERDICT_MAP[1],
    "financebench_id_00070": VERDICT_MAP[4],
    "financebench_id_00080": VERDICT_MAP[1],
    "financebench_id_00206": VERDICT_MAP[1],
    "financebench_id_00215": VERDICT_MAP[2],
    "financebench_id_00283": VERDICT_MAP[3],
    "financebench_id_00288": VERDICT_MAP[4],
    "financebench_id_00299": VERDICT_MAP[4],
    "financebench_id_00302": VERDICT_MAP[4],
    "financebench_id_00382": VERDICT_MAP[2],
}

for fid, verdict in verdicts_dict.items():
    answers_df.loc[answers_df["financebench_id"] == fid, "verdict"] = verdict

answers_df.to_excel(TASK_1_FILENAME, index=False)

In [12]:
answers_df.head(2)

,financebench_id,question_type,question,naive_answer,ground_truth,verdict
0,financebench_id_00005,domain-relevant,Does Corning have positive working capital bas...,"Based on Corning's FY2022 data, the company ha...",Yes. Corning had a positive working capital am...,correct
1,financebench_id_00070,domain-relevant,Does American Water Works have positive workin...,I don't know the specific details of American ...,"No, American Water Works had negative working ...",refused


#### Questions:

1. Cases where the model **refused** or asked for more information - why?
- In `novel-generated` Q.s, the model refused to answer (not enough knowledge) 3 times as opposed to only once for the given `domain-relevant` Q.s.<br>
I don't see a reason why sometimes it refuses to answer while sometime it hallucinates some answer...<br>
However, it DOES refuse because we told it in the prompt ('If you don't know the answer, say "I don't know"') - and it really does not have any relevant information for any of these questions.

2. Cases where the model **answered confidently** - spot-check against the ground truth. Is the answer correct? Partially correct? Totally wrong (hallucinated)?
- Overall, the model is confident with its answers, regardless of correctness. Even when it refuses to answer - it explains why it cannot answer. 
The one time it was completely wrong was when we asked it for a specific number ("How much ... **in USD million?**") - it just output some random number and was therefore wrong ("$12 billion"). For other questions, it was more like 50-50% ("Does Corning have positive working capital?") - and its best chance to succeed is to just say "yes"/"no". But it's worth as guessing.<br>
This is not surprising because that is the behaviour I encounter since starting using LLMs a couple of years ago - they always answer regardless of the data they have, mostly with a concrete answer, even when completely wrong. This holds also for the best models today.

3. Are there patterns by *question_type*? Do some types fail more than others?
- The `domain-relevant` Q.s are more yes/no Q.s.<br>
Therefore, with the naive answers we received - I see separation for the model's answers by the questions types - it answered this type of questions (but not necessarily correct).<br>
Regarding the `novel-generated` questions, those that are open questions (asking for an amount/specific segments names), it often refused to answer with "I don't know" - probably becasue it can't guess yes/no.<br>

---
## Task 2 - RAG Reminder

### RAG Pipeline Components

#### Indexing (Documents → Chunk + Embed → Vector Store)
**Contribution:** Transforms the raw corpus into a searchable knowledge base by splitting documents into manageable chunks and mapping each chunk to a dense vector in embedding space, so that semantic similarity search becomes possible later. This builds the vector store $D$ that "retrieval" will query against.

**Failure modes:** Poor chunking (e.g., splitting mid-sentence or using chunks too large/small) destroys semantic coherence - a chunk that spans two unrelated topics produces a messy embedding. The embedding model may be domain-mismatched (e.g., a general-purpose encoder on legal or medical text), causing near-duplicate documents to land far apart. Other issues: stale/missing documents, lost metadata (page numbers, titles), OCR errors, or inconsistent preprocessing between indexing time and query time.

**When:** Happens **once, offline** (with periodic re-indexing when the corpus changes). This is the most expensive step per document but amortized across all future queries.


#### Retrieval ($\Gamma$: User Query → Top-k Chunks)
**Contribution:** Given a query $q$, embeds it with the *same* encoder used at indexing and searches the vector store $D$ to return the top-$k$ most relevant chunks. This grounds the generator in specific, query-relevant evidence rather than relying purely on parametric memory.

**Failure modes:** Query-document vocabulary mismatch (user asks "how do I cancel?" but docs say "terminate subscription") - pure dense retrieval can miss this, which is why hybrid BM25+dense often helps. Wrong $k$: too small misses key context, too large dilutes the prompt with noise and wastes tokens. Other issues: multi-hop questions that need information spread across chunks, ambiguous queries, or an embedding mismatch between query-time and index-time encoders.

**When:** **Per query** - runs on every user request. Latency-sensitive, so ANN indices (FAISS, HNSW) are typically used instead of exact search.


#### Generation ($\Theta$: Query + Retrieved Chunks → Answer)
**Contribution:** An LLM consumes the query plus retrieved context and synthesizes a grounded natural-language answer, ideally citing or quoting the retrieved evidence. This is where retrieved facts become a user-facing response.

**Failure modes:** **Hallucination** even with correct context (the model ignores retrieved chunks and invents facts), or the opposite - the retrieved context *is* wrong/irrelevant and the model faithfully parrots it ("garbage in, garbage out"). Prompt-budget issues: context gets truncated and the crucial chunk is dropped. Also: "lost in the middle" (LLMs under-attend to mid-context chunks, as Yuval stated in class), stale context not reflecting the latest query intent, or tone/format drift from the system prompt.

**When:** **Per query** - one (or more) LLM calls per user request. Usually the dominant cost/latency component of the pipeline.

---
## Task 3 - Embed Documents

In [13]:
# Task 3
doc_rows = df.drop_duplicates(subset=["doc_name"], keep="first")

embeddings = HuggingFaceEmbeddings(
    model_name=EMBED_MODEL,
    encode_kwargs={"normalize_embeddings": True},
)

if os.path.isdir(VECTORSTORE_DIR):
    vectorstore = FAISS.load_local(
        VECTORSTORE_DIR,
        embeddings,
        allow_dangerous_deserialization=True,
    )
    print(
        f"Loaded FAISS index from {VECTORSTORE_DIR!r} ({vectorstore.index.ntotal} vectors)"
    )
else:
    # Load only PDFs for doc_name values in the dataset; metadata on each page before split
    all_pages = []

    for _, row in doc_rows.iterrows():
        loader = PyPDFLoader(row["doc_link"])
        pages = loader.load()

        for page_number, doc in enumerate(pages):
            doc.metadata = doc.metadata or {}
            doc.metadata["doc_name"] = row["doc_name"]
            doc.metadata["company"] = row["company"]
            doc.metadata["doc_period"] = row["doc_period"]
            doc.metadata["page_number"] = (
                page_number  # 0-indexed to match the dataset's evidence_page_num
            )
            all_pages.append(doc)

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=150,
    )
    chunks = text_splitter.split_documents(all_pages)
    vectorstore = FAISS.from_documents(chunks, embeddings)
    vectorstore.save_local(VECTORSTORE_DIR)
    print(
        f"Built and saved FAISS index to {VECTORSTORE_DIR!r} ({vectorstore.index.ntotal} vectors)"
    )

/var/folders/4p/3v7c85y55qdc7jt3wv838n700000gp/T/ipykernel_3823/2973635661.py:4: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loaded FAISS index from 'financebench_rag_faiss' (24218 vectors)


In [14]:
def check_retrieval(
    row: pd.Series, vectorstore, top_k: int = 5, verbose: bool = True
) -> dict:
    """Retrieve top-k chunks for a dataset row and check doc, page, and evidence overlap."""
    query = row["question"]
    expected_doc = row["doc_name"]

    evidence_texts, evidence_pages = [], set()
    for item in row.get("evidence", []):
        if isinstance(item, dict):
            txt = item.get("evidence_text", "")
            if txt:
                evidence_texts.append(txt)
            pg = item.get("evidence_page_num")
            if pg is not None:
                evidence_pages.add(int(pg))

    retrieved = vectorstore.similarity_search(query, k=top_k)

    doc_hits = [d for d in retrieved if d.metadata.get("doc_name") == expected_doc]
    page_hits = [
        d for d in retrieved if d.metadata.get("page_number") in evidence_pages
    ]
    overlap_hits = [
        d
        for d in retrieved
        if any(
            ev.lower()[:180] in d.page_content.lower()
            or d.page_content.lower()[:180] in ev.lower()
            for ev in evidence_texts
            if ev
        )
    ]

    result = {
        "question": query,
        "expected_doc": expected_doc,
        "evidence_pages": sorted(evidence_pages),
        "doc_match": len(doc_hits),
        "page_match": len(page_hits),
        "evidence_overlap": len(overlap_hits),
        "top_k": top_k,
        "retrieved": retrieved,
    }

    if verbose:
        print("=" * 100)
        print(f"Q: {query}")
        print(f"Expected doc_name: {expected_doc}")
        print(f"Expected evidence pages: {result['evidence_pages'] or 'N/A'}")
        print(
            f"Doc match:          {'YES' if doc_hits else 'NO'} ({len(doc_hits)}/{top_k})"
        )
        print(
            f"Evidence overlap:   {'YES' if overlap_hits else 'NO'} ({len(overlap_hits)}/{top_k})"
        )
        print(
            f"Page match:         {'YES' if page_hits else 'NO'} ({len(page_hits)}/{top_k})"
        )
        print("Top-k retrieved chunks:")
        for rank, d in enumerate(retrieved, start=1):
            print(
                f"  {rank}. {d.metadata.get('doc_name')} | page={d.metadata.get('page_number')}"
            )
            print(f"     {d.page_content[:170].replace(chr(10), ' ')}...")
        print()

    return result


# Task 3 retrieval check on 3 sample questions
retrieval_check = []

for _, row in df.head(3).iterrows():
    retrieval_check_row = check_retrieval(row, vectorstore)
    retrieval_check.append(retrieval_check_row)

retrieval_check = pd.DataFrame(retrieval_check)

Q: Does Corning have positive working capital based on FY2022 data? If working capital is not a useful or relevant metric for this company, then please state that and explain why.
Expected doc_name: CORNING_2022_10K
Expected evidence pages: [59]
Doc match:          YES (3/5)
Evidence overlap:   NO (0/5)
Page match:         NO (0/5)
Top-k retrieved chunks:
  1. CORNING_2022_10K | page=101
     (1) Corning obtained a controlling interest in HSG during the third quarter of 2020 and has consolidated results in Hemlock and Emerging Growth Businesses since September...
  2. CORNING_2022_10K | page=102
     (1) Corning obtained a controlling interest in HSG during the third quarter of 2020 and has consolidated results in Hemlock and Emerging Growth Businesses since September...
  3. 3M_2022_10K | page=37
     not defined under U.S. generally accepted accounting principles and may not be computed the same as similarly titled measures used by other companies. The Company defines...
  4. 3M_2023

#### Task 3 - Retrieval Observations

**Right document?** Mostly yes - the 3 queries returned **3/5, 4/5, and 5/5 chunks** from the correct company's filing (12/15 overall). This makes sense because the question itself mentions the company name, and that name also appears in the chunks, so the embeddings can easily match them. The misses mostly come from cross-company chunks that talk about "working capital" in general (e.g. 3M's non-GAAP definition page) and get pulled in because that phrase dominates the query.

**Right evidence text?** No - **0/5** for all 3 questions. I opened the actual PDFs on the evidence pages and found they are mostly balance-sheet tables full of numbers. When we split these pages into 1000-char chunks, the tables get broken apart and lose their structure. The retriever pulls chunks that talk *about* the right topics, but not the exact table rows the dataset annotators marked as evidence.

**Right page?** Mostly no - only **1 out of 15** retrieved chunks came from an expected evidence page (American Water Works, page 81). The balance-sheet pages are heavy on numbers and light on regular sentences, so the embedding model doesn't "understand" them well. Instead it prefers pages with more natural language, like company overviews or notes to financial statements, that *mention* working capital in words.

**Takeaway:** The retriever generally finds the right document, but struggles to find the right *page* - especially when the answer lives in a numeric table rather than a text paragraph. This is a known limitation of dense (embedding-based) retrieval. Possible improvements: combining keyword search (BM25) with embeddings, using table-aware chunking, or first filtering by document and then searching within it.

---
## Task 4 - Building a RAG Pipeline

In [15]:
def retrieve(query: str, k: int = 4) -> List[Dict]:
    """Embed `query` and return top-k chunks from the FAISS vector store,
    each as a flat dict with its text + key metadata (doc_name, page_number)."""
    docs = vectorstore.similarity_search(query, k=k)
    return [
        {
            "doc_name": d.metadata.get("doc_name"),
            "page_number": d.metadata.get("page_number"),
            "company": d.metadata.get("company"),
            "doc_period": d.metadata.get("doc_period"),
            "text": d.page_content,
        }
        for d in docs
    ]


# quick smoke test
sample_query = df.iloc[0]["question"]
sample_chunks = retrieve(sample_query, k=4)

print(f"Q: {sample_query}\n")

for i, c in enumerate(sample_chunks, 1):
    print(f"[{i}] {c['doc_name']} | page={c['page_number']}")
    print(f"    {c['text'][:160].replace(chr(10), ' ')}...\n")

Q: Does Corning have positive working capital based on FY2022 data? If working capital is not a useful or relevant metric for this company, then please state that and explain why.

[1] CORNING_2022_10K | page=101
    (1) Corning obtained a controlling interest in HSG during the third quarter of 2020 and has consolidated results in Hemlock and Emerging Growth Businesses since...

[2] CORNING_2022_10K | page=102
    (1) Corning obtained a controlling interest in HSG during the third quarter of 2020 and has consolidated results in Hemlock and Emerging Growth Businesses since...

[3] 3M_2022_10K | page=37
    not defined under U.S. generally accepted accounting principles and may not be computed the same as similarly titled measures used by other companies. The Compa...

[4] 3M_2023Q2_10Q | page=70
    Current assets $ 15,754 $ 14,688 $ 1,066  Less: Current liabilities 10,936 9,523 1,413  Working capital (non-GAAP measure) $ 4,818 $ 5,165 $ (347) Various asset...



In [16]:
SYSTEM_PROMPT = """You are a careful financial-document assistant.

Rules:
- Answer ONLY using facts that appear in the provided CONTEXT below.
- If the CONTEXT does not contain the answer, reply exactly:
  "The provided context does not contain the answer."
  Do not guess and do not use outside knowledge.
- Keep answers concise (1-4 sentences).
- Cite the source document for every fact you use, in the form (doc_name, page N).
  If multiple sources support a fact, cite all of them.
"""


def build_user_prompt(query: str, chunks: List[Dict]) -> str:
    """Format retrieved chunks into a single prompt block, with clear
    separators and doc_name/page metadata so the model can cite sources.
    Handles empty retrieval explicitly."""
    if not chunks:
        context_block = "No relevant context was retrieved."
    else:
        parts = []
        for i, c in enumerate(chunks, start=1):
            header = (
                f"--- Source {i} | doc_name={c['doc_name']} "
                f"| page={c['page_number']} ---"
            )
            parts.append(f"{header}\n{c['text']}")
        context_block = "\n\n".join(parts)

    return f"CONTEXT:\n{context_block}\n\nQUESTION: {query}\n\nANSWER:"


# inspect the prompt on our sample query
print(build_user_prompt(sample_query, sample_chunks)[:1200])
print("...")

CONTEXT:
--- Source 1 | doc_name=CORNING_2022_10K | page=101 ---
(1) Corning obtained a controlling interest in HSG during the third quarter of 2020 and has consolidated results in Hemlock and Emerging Growth Businesses since September 9, 2020.  Refer to Note 3 (HSGTransactions and Acquisitions) in the notes to the consolidated financial statements for additional information.(2) Depreciation expense for Corning’s reportable segments and Hemlock and Emerging Growth Businesses includes an allocation of depreciation of corporate property not specifically identifiable to asegment.(3) Research, development and engineering expenses include direct project spending that is identifiable to a segment.(4) Income tax (provision) benefit reflects a tax rate of 21%.(5) Segment assets include inventory, accounts receivable, property, plant and equipment, net of accumulated depreciation, and associated equity companies.
 102

--- Source 2 | doc_name=CORNING_2022_10K | page=102 ---
(1) Corning obtained

In [17]:
def generate(system_prompt: str, user_prompt: str, max_tokens: int = 400) -> str:
    """Call the generation model with a system + user prompt and return
    the stripped answer string."""
    response = client.chat.completions.create(
        model=RAG_MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        temperature=0.1,
        max_tokens=max_tokens,
    )
    return response.choices[0].message.content.strip()


# smoke test: end-to-end on the sample query, manually wiring the 3 pieces
_user_prompt = build_user_prompt(sample_query, sample_chunks)
_answer = generate(SYSTEM_PROMPT, _user_prompt)
print(f"Q: {sample_query}\n")
print(f"A: {_answer}")

Q: Does Corning have positive working capital based on FY2022 data? If working capital is not a useful or relevant metric for this company, then please state that and explain why.

A: The provided context does not contain the answer.


In [18]:
def answer_with_rag(query: str, k: int = 4) -> dict:
    """End-to-end RAG: retrieve top-k chunks from the FAISS store and
    generate a grounded, cited answer with Llama-3.3-70B-Instruct.

    Returns a dict with:
      - answer (str): the generation model's final answer.
      - retrieved_chunks (list[dict]): the chunks used as context, each with
        its doc_name, page_number, and text (plus company/doc_period for
        convenience).
    """
    retrieved_chunks = retrieve(query, k=k)
    user_prompt = build_user_prompt(query, retrieved_chunks)
    answer = generate(SYSTEM_PROMPT, user_prompt)
    return {
        "answer": answer,
        "retrieved_chunks": retrieved_chunks,
    }


# sanity test on a couple of questions from our filtered set
for _, row in df.head(2).iterrows():
    q = row["question"]
    result = answer_with_rag(q, k=4)
    print("=" * 100)
    print(f"Q: {q}")
    print(f"\nA: {result['answer']}")
    print(f"\nRetrieved {len(result['retrieved_chunks'])} chunks:")
    for i, c in enumerate(result["retrieved_chunks"], 1):
        print(f"  [{i}] {c['doc_name']} | page={c['page_number']}")
    print()

Q: Does Corning have positive working capital based on FY2022 data? If working capital is not a useful or relevant metric for this company, then please state that and explain why.

A: The provided context does not contain the answer.

Retrieved 4 chunks:
  [1] CORNING_2022_10K | page=101
  [2] CORNING_2022_10K | page=102
  [3] 3M_2022_10K | page=37
  [4] 3M_2023Q2_10Q | page=70

Q: Does American Water Works have positive working capital based on FY2022 data? If working capital is not a useful or relevant metric for this company, then please state that and explain why.

A: The provided context does not contain the answer.

Retrieved 4 chunks:
  [1] AMERICANWATERWORKS_2022_10K | page=81
  [2] AMERICANWATERWORKS_2022_10K | page=139
  [3] AMERICANWATERWORKS_2022_10K | page=144
  [4] 3M_2022_10K | page=37



---
## Task 5 - Run and Compare

In [19]:
# Run the same 10 questions from Task 1 through the RAG pipeline,
# and save a side-by-side comparison to xlsx.

TASK_5_FILENAME = "assignment2_run_and_compare.xlsx"
TASK_5_COLUMNS = [
    "financebench_id",
    "question_type",
    "question",
    "naive_answer",
    "RAG_answer",
    "ground_truth",
]

if os.path.exists(TASK_5_FILENAME):
    print("Loading existing results...")
    compare_df = pd.read_excel(TASK_5_FILENAME)
    # rebuild the retrieved chunks separately from the cached xlsx (not persisted)
    compare_df["_rag_retrieved"] = [[] for _ in range(len(compare_df))]
else:
    rows = []
    for _, row in answers_df.iterrows():
        result = answer_with_rag(row["question"], k=4)
        rows.append(
            {
                "financebench_id": row["financebench_id"],
                "question_type": row["question_type"],
                "question": row["question"],
                "naive_answer": row["naive_answer"],
                "RAG_answer": result["answer"],
                "ground_truth": row["ground_truth"],
                "_rag_retrieved": result["retrieved_chunks"],
            }
        )
        time.sleep(1.5)  # rate limiting

    compare_df = pd.DataFrame(rows)
    # persist only the required columns
    compare_df[TASK_5_COLUMNS].to_excel(TASK_5_FILENAME, index=False)
    print(f"Saved {TASK_5_FILENAME}")

compare_df.head(1)

Loading existing results...


,financebench_id,question_type,question,naive_answer,RAG_answer,ground_truth,_rag_retrieved
0,financebench_id_00005,domain-relevant,Does Corning have positive working capital bas...,"Based on Corning's FY2022 data, the company ha...",The provided context does not contain the answer.,Yes. Corning had a positive working capital am...,[]


In [20]:
# Side-by-side display of naive vs RAG vs ground truth for all 10 questions.
# Also shows the retrieved (doc_name, page) pairs so we can see *why* RAG
# answered the way it did when we write the discussion.


def _fmt(text: str, width: int = 110) -> str:
    """Wrap long strings so they render nicely in notebook output."""
    import textwrap

    return "\n     ".join(textwrap.wrap(str(text), width=width)) or "(empty)"


for _, row in compare_df.iterrows():
    print("=" * 120)
    print(f"[{row['financebench_id']}] ({row['question_type']})")
    print(f"Q:   {_fmt(row['question'])}")
    print(f"\nNAIVE:   {_fmt(row['naive_answer'])}")
    print(f"\nRAG:     {_fmt(row['RAG_answer'])}")
    print(f"\nGT:      {_fmt(row['ground_truth'])}")
    retrieved = row.get("_rag_retrieved") or []

    if retrieved:
        pages = ", ".join(f"{c['doc_name']}#p{c['page_number']}" for c in retrieved)
        print(f"\nRetrieved: {pages}")

    print()

[financebench_id_00005] (domain-relevant)
Q:   Does Corning have positive working capital based on FY2022 data? If working capital is not a useful or
     relevant metric for this company, then please state that and explain why.

NAIVE:   Based on Corning's FY2022 data, the company has positive working capital. Working capital is a relevant metric
     for Corning, as it is a manufacturing company that requires significant investments in inventory, accounts
     receivable, and accounts payable. According to Corning's FY2022 financial statements, the company's current
     assets exceed its current liabilities, indicating positive working capital. This suggests that Corning has
     sufficient liquidity to meet its short-term obligations.

RAG:     The provided context does not contain the answer.

GT:      Yes. Corning had a positive working capital amount of $831 million by FY 2022 close. This answer considers
     only operating current assets and current liabilities that were clear

### RAG vs. Naive: Analysis of the 10 Sample Comparisons

Looking at the results, here's the breakdown:

#### 1. Where RAG Helped (grounded answers over refusals/hallucinations)

| ID | Type | Why RAG won |
|---|---|---|
| `00215` Verizon capex | domain-relevant | Both right, but RAG added a verifiable citation (`VERIZON_2022_10K, page 22`) with $23.1B figure vs. naive's un-cited $22.6B |
| `00288` Best Buy cash drop | novel-generated | Naive refused outright; RAG pulled the exact numbers ($1,874 → $1,093) from `BESTBUY_2024Q2_10Q, page 19` |
| `00382` MGM EBITDAR by region | novel-generated | Both named Las Vegas, but RAG added the specific $3.142B figure and earnings-release citation |
| `00283` Pfizer/Upjohn spinoff | novel-generated | *Partial win* - naive hallucinated "$12B", RAG correctly refused. Neither produced GT ($77.78M), but RAG at least avoided a confidently wrong answer |

#### 2. Where RAG Hurt (naive memorization beat retrieval)

| ID | Type | What went wrong |
|---|---|---|
| `00005` Corning WC | domain-relevant | Naive answered "positive" (matches GT); RAG refused - retrieval likely missed Corning's balance-sheet chunk |
| `00080` PayPal WC | domain-relevant | Same pattern - naive's memorized "positive" matched GT; RAG said "context does not contain the answer" |
| `00206` JPM gross margins | domain-relevant | Naive gave a textbook-perfect explanation (gross margin irrelevant for a bank - matches GT). RAG refused. The question is *conceptual*, so retrieval couldn't help |

In all three, RAG's strict "stick to retrieved context" prompting turned a correct memorized answer into an unnecessary refusal.

#### 3. Where Both Failed

- `00070` American Water Works WC - RAG retrieved only the liabilities page (`page 81`), missing current assets; naive refused.
- `00299` JPM Q1 2021 segment revenue - both refused; specific quarterly segment data likely not in the retrieved chunks.
- `00302` Pfizer PPNE FY20→FY21 - both refused.

#### 4. Patterns by `question_type`

Tallying the 10 cases:

| Type | RAG helped | RAG hurt | Both failed | Tie |
|---|---|---|---|---|
| domain-relevant (5) | 1 (`00215`) | 3 (`00005`, `00080`, `00206`) | 1 (`00070`) | 0 |
| novel-generated (5) | 2 (`00288`, `00382`) + 1 partial (`00283`) | 0 | 2 (`00299`, `00302`) | 0 |

**RAG hurts more on `domain-relevant`, helps more on `novel-generated`.**

##### Hypothesis - why the asymmetry?

1. **`domain-relevant` questions are answerable from pretraining.** Things like "does PayPal/Corning have positive working capital?" or "are gross margins relevant for JPM?" are general-knowledge / conceptual questions the base LLM already knows. RAG's grounding prompt ("answer only from context") then *penalizes* the model: if the retrieved chunks don't literally state the figure, it refuses - even when it "knows" the answer. Retrieval thus acts as a gate that filters out correct memorized answers.

2. **`novel-generated` questions are specific, filing-level lookups** (exact cash balances, exact EBITDAR, exact segment revenue). The naive model can't memorize these - it either refuses or fabricates. RAG has a real shot because the answer literally lives in a chunk, and when retrieval works (`00288`, `00382`) the grounded citation is a big win.

3. **Retrieval recall is the bottleneck on both sides.** When RAG hurts domain-relevant questions, it's usually because the retriever surfaced *some* chunks from the right filing but not the one containing the balance-sheet figure, so the model conservatively abstains. Improving chunking (e.g. table-aware splitting) or boosting top-k for financial-statement pages would likely flip several of the "RAG hurt" cases into wins without regressing the "RAG helped" cases.

##### Summary

- RAG's gain shows up where it should: specific, filing-grounded facts with citations.
- The cost is over-conservative refusals on conceptual/common-knowledge questions. A *hybrid* policy - "answer from context if present, otherwise fall back to parametric knowledge with a clear disclaimer" - would likely dominate the current pure-RAG prompt on this mix.

---
## Task 6 - Evaluation

We evaluate the RAG pipeline along three complementary axes, each probing a different failure mode:

| Metric | What it measures | How we compute it |
|---|---|---|
| **Correctness** | Does the final answer match the ground truth? | LLM-as-judge with **DeepSeek-V3.2** (different from generator) → binary verdict + 1-sentence justification |
| **Faithfulness** | Does the answer stay grounded in retrieved context (no hallucinations)? | Ragas `Faithfulness` metric on the first 20 questions |
| **Page-hit@k** | Did retrieval surface the evidence page? | For each question, check if any of the top-k chunks has a `page_number` in `evidence_page_num`. Report for k ∈ {1, 3, 5} |

The steps:
1. Run **all** questions through the RAG pipeline (cached).
2. Score each answer with the **correctness** judge.
3. Compute **page-hit@k** from a parallel top-5 retrieval.
4. Compute **faithfulness** (Ragas) on the first 20 questions.
5. Aggregate + save `assignment2_evaluation.xlsx`.

#### Step 1 - Run the full dataset through the RAG pipeline

We call the `answer_with_rag` we built in Task 4 on every question (~150 rows after filtering `metrics-generated`). Each call is one Llama-3.3-70B request, so we **cache** results to a pickle; re-running the cell just loads the cache.

We call it with **`k=5`** (one chunk more than the Task 4/5 default of 4) because page-hit@k needs up to 5 retrieved chunks. FAISS returns chunks in rank order, so the same top-5 set serves both purposes - no need for a second retrieval:
- `rag_answer` - the generator's output (generator now sees 5 chunks)
- `retrieved_chunks` - the 5 chunks fed to the generator, reused for both *faithfulness* and *page-hit@k* (k ∈ {1, 3, 5})

In [21]:
RAG_RUN_CACHE = "task6_rag_run.pkl"
RAG_K = 5  # one chunk more than Task 4/5 so the same retrieval serves page-hit@{1,3,5}

if os.path.exists(RAG_RUN_CACHE):
    with open(RAG_RUN_CACHE, "rb") as f:
        rag_runs = pickle.load(f)
    print(f"Loaded cached RAG runs ({len(rag_runs)} rows) from {RAG_RUN_CACHE!r}")
else:
    rag_runs = []
    for _, row in tqdm(df.iterrows(), total=len(df), desc="RAG"):
        try:
            out = answer_with_rag(row["question"], k=RAG_K)
            answer = out["answer"]
            retrieved = out["retrieved_chunks"]
        except Exception as e:
            answer = f"[ERROR] {e}"
            retrieved = []

        rag_runs.append(
            {
                "financebench_id": row["financebench_id"],
                "question": row["question"],
                "ground_truth": row["answer"],
                "evidence": row["evidence"],
                "rag_answer": answer,
                "retrieved_chunks": retrieved,
            }
        )
        time.sleep(1.0)  # gentle rate limit

    with open(RAG_RUN_CACHE, "wb") as f:
        pickle.dump(rag_runs, f)
    print(f"Saved {len(rag_runs)} rows to {RAG_RUN_CACHE!r}")

print(
    "Sample:",
    rag_runs[0]["financebench_id"],
    "->",
    rag_runs[0]["rag_answer"][:120],
    "...",
)

Loaded cached RAG runs (100 rows) from 'task6_rag_run.pkl'
Sample: financebench_id_00005 -> The provided context does not contain the answer. ...


#### Step 2 - Correctness (LLM-as-judge)

We use **DeepSeek-V3.2** (defined in `JUDGE_MODEL`) - **different** from the Llama generator - so the judge isn't scoring its own output.

The judge prompt:
- Returns a strict JSON object `{"verdict": "correct"|"incorrect", "justification": "..."}`.
- Treats **semantically equivalent** answers as correct (e.g. "$23.1B" vs. "23,100 million USD") but is strict about numbers, direction, and named entities.
- Counts an explicit "context does not contain the answer" refusal as *incorrect* when the GT gives a concrete answer - we want the correctness metric to reflect final-answer quality, not the generator's epistemic honesty (faithfulness covers that).

In [22]:
JUDGE_SYSTEM_PROMPT = """You are an impartial grader for a financial QA system.

You compare a MODEL_ANSWER to a GROUND_TRUTH answer for a QUESTION about SEC filings.

Rules:
- Judge strictly on factual alignment. Semantic equivalents are OK (e.g. "$23.1B" == "23,100 million USD"; "yes, positive" == "positive working capital").
- Numerical values must match (tolerance: rounding to the same significant figures as the GT).
- Names, directions (increase/decrease), and signs must match.
- If MODEL_ANSWER says it cannot answer / context insufficient while GROUND_TRUTH has a concrete answer, that is "incorrect".
- If GROUND_TRUTH itself says the concept is not meaningful and MODEL_ANSWER agrees, that is "correct".
- If MODEL_ANSWER reaches the same core conclusion as GROUND_TRUTH (same yes/no, same direction, same numeric value within rounding tolerance) but arrives there via different legitimate reasoning or supporting evidence, count as correct. Only mark incorrect when (a) the core conclusion disagrees, (b) a directly-answered number is wrong or hallucinated, (c) the model refuses / claims insufficient context while GT has a concrete answer, or (d) the model invents facts not in any reasonable source.

Output ONLY a compact JSON object on a single line:
{"verdict": "correct" | "incorrect", "justification": "<=1 sentence"}
Do not wrap in markdown. Do not add any other text."""


def judge_correctness(question: str, model_answer: str, ground_truth: str) -> dict:
    user_prompt = (
        f"QUESTION: {question}\n\n"
        f"GROUND_TRUTH: {ground_truth}\n\n"
        f"MODEL_ANSWER: {model_answer}\n\n"
        "Return the JSON verdict now."
    )
    resp = client.chat.completions.create(
        model=JUDGE_MODEL,
        messages=[
            {"role": "system", "content": JUDGE_SYSTEM_PROMPT},
            {"role": "user", "content": user_prompt},
        ],
        temperature=0.0,
        max_tokens=200,
    )
    raw = resp.choices[0].message.content.strip()
    # robust parse: grab the first {...} block
    m = re.search(r"\{.*\}", raw, re.DOTALL)
    if not m:
        return {"verdict": "incorrect", "justification": f"[parse-fail] {raw[:120]}"}
    try:
        obj = _json.loads(m.group(0))
        v = str(obj.get("verdict", "")).lower().strip()
        if v not in {"correct", "incorrect"}:
            v = "incorrect"
        return {"verdict": v, "justification": str(obj.get("justification", ""))[:300]}
    except Exception:
        return {"verdict": "incorrect", "justification": f"[json-fail] {raw[:120]}"}


# Smoke test on one row to make sure the judge responds sensibly.
_t = rag_runs[0]
_v = judge_correctness(_t["question"], _t["rag_answer"], _t["ground_truth"])
print(_t["financebench_id"], "->", _v)

financebench_id_00005 -> {'verdict': 'incorrect', 'justification': 'The model incorrectly states the context lacks the answer, but the ground truth provides a clear answer based on the data.'}


In [23]:
# Run the judge on all rows (cached).
CORRECTNESS_CACHE = "task6_correctness.pkl"

if os.path.exists(CORRECTNESS_CACHE):
    with open(CORRECTNESS_CACHE, "rb") as f:
        correctness = pickle.load(f)
    print(f"Loaded cached correctness ({len(correctness)} rows)")
else:
    correctness = []
    for r in tqdm(rag_runs, desc="judge"):
        v = judge_correctness(r["question"], r["rag_answer"], r["ground_truth"])
        correctness.append({"financebench_id": r["financebench_id"], **v})
        time.sleep(0.6)
    with open(CORRECTNESS_CACHE, "wb") as f:
        pickle.dump(correctness, f)
    print(f"Saved correctness for {len(correctness)} rows")

# Aggregate preview
_df_corr = pd.DataFrame(correctness)
_df_corr["correct"] = (_df_corr["verdict"] == "correct").astype(int)
print(
    f"Average correctness: {_df_corr['correct'].mean():.3f} "
    f"({_df_corr['correct'].sum()}/{len(_df_corr)})"
)
_df_corr.head(5)

Loaded cached correctness (100 rows)
Average correctness: 0.290 (29/100)


,financebench_id,verdict,justification,correct
0,financebench_id_00005,incorrect,The model refused to answer despite the ground...,0
1,financebench_id_00070,incorrect,The model refused to answer despite the ground...,0
2,financebench_id_00080,incorrect,The model incorrectly states the context lacks...,0
3,financebench_id_00206,incorrect,The model failed to state that gross margin is...,0
4,financebench_id_00215,incorrect,The model answer does not address the capital ...,0


#### Step 3 - Page-hit@k

For each row we already stored the top-5 retrieved chunks (`retrieved_chunks`, from Step 1's `k=5` call). The dataset's `evidence` column is a list of dicts; we extract each `evidence_page_num` into an **expected-page set** (multiple evidence items → multiple pages → it counts as a hit if retrieval surfaces *any* of them).

`page_hit_at_k` for a question is `1` if the first-k retrieved chunks include at least one chunk whose `page_number` is in that expected-page set, else `0`. We compute it for k ∈ {1, 3, 5}.

In [24]:
HIT_KS = [1, 3, 5]


def expected_pages(evidence_field) -> set:
    pages = set()
    if not isinstance(evidence_field, (list, tuple)):
        return pages
    for item in evidence_field:
        if isinstance(item, dict) and item.get("evidence_page_num") is not None:
            try:
                pages.add(int(item["evidence_page_num"]))
            except (TypeError, ValueError):
                pass
    return pages


def page_hits(
    retrieved: list, exp_pages: set, expected_doc: str = None, ks=HIT_KS
) -> dict:
    out = {}
    for k in ks:
        top_k = retrieved[:k]
        if expected_doc:
            top_k_pages = {
                c.get("page_number") for c in top_k if c.get("doc_name") == expected_doc
            }
        else:
            top_k_pages = {c.get("page_number") for c in top_k}
        out[f"page_hit_at_{k}"] = int(bool(top_k_pages & exp_pages)) if exp_pages else 0
    return out


_doc_name_map = dict(zip(df["financebench_id"], df["doc_name"]))

page_hit_rows = []
for r in rag_runs:
    ep = expected_pages(r["evidence"])
    exp_doc = _doc_name_map.get(r["financebench_id"])
    hits = page_hits(r["retrieved_chunks"], ep, expected_doc=exp_doc)
    page_hit_rows.append(
        {"financebench_id": r["financebench_id"], **hits, "n_evidence_pages": len(ep)}
    )

page_hit_df = pd.DataFrame(page_hit_rows)
print("Aggregate page-hit@k:")
for k in HIT_KS:
    col = f"page_hit_at_{k}"
    print(f"  k={k}: {page_hit_df[col].mean():.3f}")
page_hit_df.head(5)

Aggregate page-hit@k:
  k=1: 0.200
  k=3: 0.330
  k=5: 0.400


,financebench_id,page_hit_at_1,page_hit_at_3,page_hit_at_5,n_evidence_pages
0,financebench_id_00005,0,0,0,1
1,financebench_id_00070,1,1,1,2
2,financebench_id_00080,0,0,0,1
3,financebench_id_00206,0,0,0,1
4,financebench_id_00215,0,1,1,2


#### Step 4 - Faithfulness (Ragas)

Faithfulness asks: *do the claims in the answer follow from the retrieved context?*  Ragas does this by (1) decomposing the answer into atomic claims with an LLM, then (2) checking each claim for entailment against the provided contexts.

Setup (following the TA's correction to the original assignment note):
- Use the **sync** `OpenAI` client (reuse the global `client` from the constants cell; not `AsyncOpenAI`).
- Wrap **DeepSeek-V3.2** with `ragas.llms.llm_factory(model=JUDGE_MODEL, client=client)` so the metric uses the judge model.
- Call `.single_turn_score(sample)` (the synchronous scoring method for a `SingleTurnSample` in Ragas 0.2+).
- Run on the **first 20** questions (sorted by `financebench_id`, which our `df` already is).

In [25]:
# Per TA: use the sync OpenAI client (not AsyncOpenAI). We reuse the global
# `client` defined in the constants cell - same Nebius base URL, any model.
ragas_llm = llm_factory(model=JUDGE_MODEL, client=client)
faithfulness_metric = Faithfulness(llm=ragas_llm)
print("Ragas Faithfulness ready with model:", JUDGE_MODEL)

Ragas Faithfulness ready with model: deepseek-ai/DeepSeek-V3.2


In [26]:
FAITHFULNESS_CACHE = "task6_faithfulness.pkl"
FAITHFULNESS_N = 20

if os.path.exists(FAITHFULNESS_CACHE):
    with open(FAITHFULNESS_CACHE, "rb") as f:
        faithfulness_scores = pickle.load(f)
    print(f"Loaded cached faithfulness ({len(faithfulness_scores)} rows)")
else:
    faithfulness_scores = []
    for r in tqdm(rag_runs[:FAITHFULNESS_N], desc="faithfulness"):
        contexts = [c["text"] for c in r["retrieved_chunks"]] or [""]
        sample = SingleTurnSample(
            user_input=r["question"],
            response=r["rag_answer"],
            retrieved_contexts=contexts,
        )
        try:
            # Ragas 0.2+ sync API: single_turn_score is the synchronous
            # counterpart of single_turn_ascore. The older .score()/.ascore()
            # pair only accepts **kwargs in this version.
            score = faithfulness_metric.single_turn_score(sample)
        except Exception as e:
            print("  faithfulness error on", r["financebench_id"], ":", repr(e))
            score = float("nan")
        faithfulness_scores.append(
            {"financebench_id": r["financebench_id"], "faithfulness": float(score)}
        )

    with open(FAITHFULNESS_CACHE, "wb") as f:
        pickle.dump(faithfulness_scores, f)
    print(f"Saved faithfulness for {len(faithfulness_scores)} rows")

_fs = pd.DataFrame(faithfulness_scores)
print(
    f"Average faithfulness (n={len(_fs)}, NaNs ignored): {_fs['faithfulness'].mean(skipna=True):.3f}"
)
_fs.head(5)

Loaded cached faithfulness (20 rows)
Average faithfulness (n=20, NaNs ignored): 0.423


,financebench_id,faithfulness
0,financebench_id_00005,0.5
1,financebench_id_00070,0.0
2,financebench_id_00080,1.0
3,financebench_id_00206,0.0
4,financebench_id_00215,1.0


#### Step 5 - Assemble per-question table + aggregates

We merge the three metric tables by `financebench_id` into the final deliverable `assignment2_evaluation.xlsx`, with columns:

`financebench_id | question | correctness | faithfulness | page_hit_at_1 | page_hit_at_3 | page_hit_at_5`

Faithfulness is only defined for the first 20 rows - the remaining rows get `NaN`.

In [27]:
eval_df = pd.DataFrame(
    {
        "financebench_id": [r["financebench_id"] for r in rag_runs],
        "question": [r["question"] for r in rag_runs],
    }
)

corr_df = pd.DataFrame(correctness)
corr_df["correctness"] = (corr_df["verdict"] == "correct").astype(int)

eval_df = (
    eval_df.merge(
        corr_df[["financebench_id", "correctness"]], on="financebench_id", how="left"
    )
    .merge(pd.DataFrame(faithfulness_scores), on="financebench_id", how="left")
    .merge(
        page_hit_df[["financebench_id"] + [f"page_hit_at_{k}" for k in HIT_KS]],
        on="financebench_id",
        how="left",
    )
)

TASK_6_FILENAME = "assignment2_evaluation.xlsx"
eval_df.to_excel(TASK_6_FILENAME, index=False)
print(f"Saved {TASK_6_FILENAME} with {len(eval_df)} rows")
eval_df.head(5)

Saved assignment2_evaluation.xlsx with 100 rows


,financebench_id,question,correctness,faithfulness,page_hit_at_1,page_hit_at_3,page_hit_at_5
0,financebench_id_00005,Does Corning have positive working capital bas...,0,0.5,0,0,0
1,financebench_id_00070,Does American Water Works have positive workin...,0,0.0,1,1,1
2,financebench_id_00080,Does Paypal have positive working capital base...,0,1.0,0,0,0
3,financebench_id_00206,Are JPM's gross margins historically consisten...,0,0.0,0,0,0
4,financebench_id_00215,Is Verizon a capital intensive business based ...,0,1.0,0,1,1


In [28]:
# Aggregate numbers
print("=" * 60)
print("Task 6 - Aggregate Results")
print("=" * 60)
print(f"N questions:              {len(eval_df)}")
print(
    f"Average correctness:      {eval_df['correctness'].mean():.3f} "
    f"({eval_df['correctness'].sum()}/{len(eval_df)})"
)
print(
    f"Average faithfulness:     {eval_df['faithfulness'].mean(skipna=True):.3f} "
    f"(on {eval_df['faithfulness'].notna().sum()} rows)"
)
for k in HIT_KS:
    col = f"page_hit_at_{k}"
    print(f"Page-hit@{k}:              {eval_df[col].mean():.3f}")

Task 6 - Aggregate Results
N questions:              100
Average correctness:      0.290 (29/100)
Average faithfulness:     0.423 (on 20 rows)
Page-hit@1:              0.200
Page-hit@3:              0.330
Page-hit@5:              0.400


In [29]:
# Verify the faithfulness-by-refusal split claim.
# Refusal heuristic: literal substring match on the system prompt's refusal phrase.
with open(FAITHFULNESS_CACHE, "rb") as f:
    _faith = pickle.load(f)
with open(RAG_RUN_CACHE, "rb") as f:
    _rag = pickle.load(f)

_rag_by_id = {r["financebench_id"]: r["rag_answer"] for r in _rag}
_fs = pd.DataFrame(_faith)
_fs["rag_answer"] = _fs["financebench_id"].map(_rag_by_id)
_fs["is_refusal"] = _fs["rag_answer"].str.contains(
    "does not contain the answer", case=False, regex=False
)

print(f"Total faithfulness subset: {len(_fs)}")
print(f"  Refusals:    {int(_fs['is_refusal'].sum())}")
print(f"  Substantive: {int((~_fs['is_refusal']).sum())}\n")

print("Mean faithfulness by group:")
print(
    _fs.groupby("is_refusal")["faithfulness"]
    .agg(["count", "mean", "min", "max"])
    .round(3)
)
print()

n_zeros = int((_fs["faithfulness"] == 0).sum())
n_refusal_zeros = int(((_fs["faithfulness"] == 0) & _fs["is_refusal"]).sum())
print(f"Zero-faithfulness rows: {n_zeros}")
print(f"  of which refusals:    {n_refusal_zeros}")
print(f"  of which substantive: {n_zeros - n_refusal_zeros}")

_fs[["financebench_id", "faithfulness", "is_refusal"]]

Total faithfulness subset: 20
  Refusals:    11
  Substantive: 9

Mean faithfulness by group:
            count   mean  min  max
is_refusal                        
False           9  0.663  0.0  1.0
True           11  0.227  0.0  1.0

Zero-faithfulness rows: 9
  of which refusals:    8
  of which substantive: 1


,financebench_id,faithfulness,is_refusal
0,financebench_id_00005,0.500000,True
1,financebench_id_00070,0.000000,True
2,financebench_id_00080,1.000000,True
3,financebench_id_00206,0.000000,True
4,financebench_id_00215,1.000000,False
5,financebench_id_00216,1.000000,True
6,financebench_id_00222,0.000000,True
7,financebench_id_00283,0.750000,False
8,financebench_id_00288,0.000000,False
9,financebench_id_00299,0.000000,True


### Task 6 - Results Summary

All numbers below come from the pickled caches (`task6_rag_run.pkl`, `task6_correctness.pkl`, `task6_faithfulness.pkl`) and the per-question `assignment2_evaluation.xlsx`. The RAG pipeline (k=5, Llama-3.3-70B generation, BGE-small embeddings, FAISS) was run on all **100** questions after dropping *metrics-generated* (split 50/50 between *domain-relevant* and *novel-generated*).

#### Aggregate numbers

| Metric | Value | Notes |
|---|---|---|
| Correctness (judge: DeepSeek-V3.2, binary) | **29 / 100 = 0.290** | 71 judged incorrect |
| Faithfulness (Ragas, first 20) | **0.423** | distribution: 9 × 0.0, 3 × 0.5, 1 × 0.67, 2 × 0.75, 1 × 0.8, 4 × 1.0 |
| Page-hit@1 | **0.20** | 20 / 100 |
| Page-hit@3 | **0.33** | 33 / 100 |
| Page-hit@5 | **0.40** | 40 / 100 |

#### Breakdown by `question_type`

Consistent with Task 5's qualitative read: **RAG does better on *novel-generated* questions** because they need a specific filing lookup and retrieval is more likely to surface the right page. *Domain-relevant* questions are often conceptual yes/no ("is X capital intensive?", "does Y have positive working capital?") where the evidence pages are balance-sheet tables - exactly the content our dense retriever struggles with (cf. Task 3 observations). The exact per-type splits are in the diagnostics tables printed below (in the Task 7 experiment cells).

#### Retrieval is the bottleneck

The diagnostics tables above cross-tabulate correctness against `page_hit_at_5`. The pattern is stark:

- **When retrieval succeeds** (hit@5 = 1), conditional correctness is ~55%.
- **When retrieval misses** (hit@5 = 0), conditional correctness drops to ~12%.
- Retrieval roughly **quadruples** the chance of a correct answer. The pipeline's hard ceiling is whatever fraction of questions we can surface the right page for (currently 40%).

Some questions are judged correct even without a page hit - mostly segment-level questions (MGM, AMD, J&J) where the same figure appears on multiple adjacent pages (segment tables, discussion sections), so the retriever landed on a *neighboring* page that still contained the answer text.

#### Refusals dominate the failure mode

58 / 100 RAG answers contain the literal string "The provided context does not contain the answer." (the system-prompt refusal phrase).

| | count | judged correct |
|---|---|---|
| Refusal | 58 | 0 / 58 |
| Concrete answer | 42 | 29 / 42 = **69%** |

**When the model commits to an answer, it's right ~69% of the time.** All the correctness loss comes from retrieval misses → model refuses → judge marks incorrect. This is our strict-grounding prompt working as designed: the model only answers when it has support, and when it does, it's reliable.

#### Faithfulness caveats (n=20 is small)

On the 20-row subsample, 9 answers scored 0.0 - 8 of them are refusals. Refusals average 0.227 faithfulness (n=11) while substantive answers average 0.663 (n=9), confirming that refusals drag down the overall mean (0.423) and that when the model actually commits, it stays meaningfully more grounded. Caveat: the refusal → 0 pattern isn't absolute (3 of 11 refusals scored > 0, two even hit 1.0), so Ragas' treatment of refusal phrasing has variance.

#### Known judge miscalibration

`financebench_id_00215` (Verizon capital-intensive) was judged **incorrect** with justification *"the model answer does not address the capital intensity ratio provided in the ground truth and instead focuses on capital expenditures"*. Both GT and the RAG answer concluded "yes, capital intensive" - they differ only in which financial definition they invoke (asset/revenue ratio vs. capex magnitude). Both are legitimate. The judge prompt rewards *metric-match* rather than *conclusion-match*, which biases correctness downward on domain-relevant yes/no questions. Task 1's manual grading marked the naive version of this question as "partially correct" for the same reason - the binary 0/1 rubric can't represent it cleanly.

A one-rule prompt tweak ("Judge the conclusion, not the reasoning path; accept alternative valid reasoning that reaches the same conclusion") would flip 00215 and probably a handful of similar cases. The present numbers are therefore a conservative lower bound on correctness.

#### What this points at for Task 7

1. **Retrieval is the cheapest place to get gains.** Hit@5 = 40% and conditional correctness above that line is ~55%. Moving hit@5 up by any means (reranker, bigger k, BM25-hybrid, table-aware chunking) should directly lift correctness.
2. **The generation prompt is already near-optimal for this metric mix** (69% conditional accuracy on commitments). Relaxing it to "answer from context if present, else fall back to parametric knowledge with a disclaimer" would trade faithfulness for correctness - worth exploring as a separate experiment.
3. **Chunk size 1000** produced tables that split mid-row on balance-sheet pages (the pages our retriever misses most). Varying chunk size / overlap is likely to move page-hit@k noticeably.
4. **Judge prompt variants** are themselves an experiment worth running - the gap between the strict and lenient versions is probably 5–10 percentage points of correctness on this distribution.

---
## Task 7 - Improvement Cycles

In this section, we run three controlled experiments (one change at a time) against the Task 6 baseline. Task 6 cells and cached outputs remain unchanged; Task 7 uses additive code and separate caches.

Protocol per experiment:
1. State a hypothesis.
2. Change one pipeline component.
3. Re-run correctness, faithfulness (same first-20 subset), and page-hit@k.
4. Compare against baseline with code-generated tables.

All conclusions are deferred until after running the code and reviewing the printed metrics.

In [30]:
from dataclasses import dataclass
from typing import Callable, Optional


@dataclass
class PipelineConfig:
    name: str
    change: str
    retrieve_fn: Optional[Callable[[str, int], List[Dict]]] = None
    system_prompt: str = SYSTEM_PROMPT
    generation_model: str = RAG_MODEL
    k: int = RAG_K


def _doc_to_chunk_dict(doc) -> Dict:
    return {
        "doc_name": doc.metadata.get("doc_name"),
        "page_number": doc.metadata.get("page_number"),
        "company": doc.metadata.get("company"),
        "doc_period": doc.metadata.get("doc_period"),
        "text": doc.page_content,
    }


def _generate_with_model(
    system_prompt: str,
    user_prompt: str,
    model: str,
    max_tokens: int = 400,
) -> str:
    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        temperature=0.1,
        max_tokens=max_tokens,
    )
    return response.choices[0].message.content.strip()


def _load_or_run_pickle(path: str, builder: Callable[[], list], label: str) -> list:
    if os.path.exists(path):
        with open(path, "rb") as f:
            data = pickle.load(f)
        print(f"Loaded {label} ({len(data)} rows) from {path!r}")
        return data

    data = builder()
    with open(path, "wb") as f:
        pickle.dump(data, f)
    print(f"Saved {label} ({len(data)} rows) to {path!r}")
    return data


def _run_rag(df_source: pd.DataFrame, cfg: PipelineConfig, retrieve_fn) -> list:
    rows = []
    for _, row in tqdm(
        df_source.iterrows(), total=len(df_source), desc=f"RAG:{cfg.name}"
    ):
        try:
            retrieved = retrieve_fn(row["question"], k=cfg.k)
            user_prompt = build_user_prompt(row["question"], retrieved)
            answer = _generate_with_model(
                system_prompt=cfg.system_prompt,
                user_prompt=user_prompt,
                model=cfg.generation_model,
            )
        except Exception as e:
            answer = f"[ERROR] {e}"
            retrieved = []

        rows.append(
            {
                "financebench_id": row["financebench_id"],
                "question": row["question"],
                "question_type": row["question_type"],
                "ground_truth": row["answer"],
                "evidence": row["evidence"],
                "rag_answer": answer,
                "retrieved_chunks": retrieved,
            }
        )
        time.sleep(1.0)
    return rows


def _run_correctness(runs: list, sleep_s: float = 0.6) -> list:
    rows = []
    for r in tqdm(runs, desc="judge"):
        verdict = judge_correctness(r["question"], r["rag_answer"], r["ground_truth"])
        rows.append({"financebench_id": r["financebench_id"], **verdict})
        time.sleep(sleep_s)
    return rows


def _run_faithfulness(runs_subset: list) -> list:
    rows = []
    for r in tqdm(runs_subset, desc="faithfulness"):
        contexts = [c["text"] for c in r["retrieved_chunks"]] or [""]
        sample = SingleTurnSample(
            user_input=r["question"],
            response=r["rag_answer"],
            retrieved_contexts=contexts,
        )
        try:
            score = faithfulness_metric.single_turn_score(sample)
        except Exception as e:
            print("  faithfulness error on", r["financebench_id"], ":", repr(e))
            score = float("nan")

        rows.append(
            {
                "financebench_id": r["financebench_id"],
                "faithfulness": float(score),
            }
        )
    return rows


def _build_page_hit_df(runs: list, df_source: pd.DataFrame = None) -> pd.DataFrame:
    doc_map = (
        dict(zip(df["financebench_id"], df["doc_name"]))
        if df_source is None
        else dict(zip(df_source["financebench_id"], df_source["doc_name"]))
    )
    rows = []
    for r in runs:
        ep = expected_pages(r["evidence"])
        exp_doc = doc_map.get(r["financebench_id"])
        hits = page_hits(r["retrieved_chunks"], ep, expected_doc=exp_doc)
        rows.append(
            {
                "financebench_id": r["financebench_id"],
                "question_type": r.get("question_type"),
                **hits,
            }
        )
    return pd.DataFrame(rows)


def _aggregate_metrics(
    correctness_rows: list, faithfulness_rows: list, page_hit_df: pd.DataFrame
) -> Dict:
    corr_df = pd.DataFrame(correctness_rows)
    corr_df["correctness"] = (corr_df["verdict"] == "correct").astype(int)

    faith_df = pd.DataFrame(faithfulness_rows)

    out = {
        "correctness": float(corr_df["correctness"].mean()),
        "faithfulness": float(faith_df["faithfulness"].mean(skipna=True)),
    }
    for k in HIT_KS:
        out[f"page_hit_at_{k}"] = float(page_hit_df[f"page_hit_at_{k}"].mean())
    return out


def run_experiment(cfg: PipelineConfig) -> Dict:
    rag_cache = f"task7_{cfg.name}_rag_run.pkl"
    corr_cache = f"task7_{cfg.name}_correctness.pkl"
    faith_cache = f"task7_{cfg.name}_faithfulness.pkl"

    retrieve_fn = cfg.retrieve_fn or retrieve

    runs = _load_or_run_pickle(
        rag_cache,
        lambda: _run_rag(df, cfg, retrieve_fn),
        label=f"task7 rag ({cfg.name})",
    )
    correctness_rows = _load_or_run_pickle(
        corr_cache,
        lambda: _run_correctness(runs),
        label=f"task7 correctness ({cfg.name})",
    )
    faithfulness_rows = _load_or_run_pickle(
        faith_cache,
        lambda: _run_faithfulness(runs[:FAITHFULNESS_N]),
        label=f"task7 faithfulness ({cfg.name})",
    )

    page_hit_df = _build_page_hit_df(runs)
    metrics = _aggregate_metrics(correctness_rows, faithfulness_rows, page_hit_df)

    return {
        "experiment": cfg.name,
        "change": cfg.change,
        **metrics,
        "cache_paths": {
            "rag": rag_cache,
            "correctness": corr_cache,
            "faithfulness": faith_cache,
        },
    }


def load_baseline_from_task6_caches() -> Dict:
    rag_cache = "task6_rag_run.pkl"
    corr_cache = "task6_correctness.pkl"
    faith_cache = "task6_faithfulness.pkl"

    with open(rag_cache, "rb") as f:
        rag_runs = pickle.load(f)
    with open(corr_cache, "rb") as f:
        correctness_rows = pickle.load(f)
    with open(faith_cache, "rb") as f:
        faithfulness_rows = pickle.load(f)

    page_hit_df = _build_page_hit_df(rag_runs)
    metrics = _aggregate_metrics(correctness_rows, faithfulness_rows, page_hit_df)

    return {
        "experiment": "baseline",
        "change": "Task 6 baseline (k=5, chunk=1000, strict prompt)",
        **metrics,
        "cache_paths": {
            "rag": rag_cache,
            "correctness": corr_cache,
            "faithfulness": faith_cache,
        },
    }


def _safe_mean(series: pd.Series) -> float:
    if series.empty:
        return float("nan")
    return float(series.mean())


def build_diagnostics(
    results: Dict[str, Dict], df_source: pd.DataFrame
) -> pd.DataFrame:
    question_type_map = dict(
        zip(df_source["financebench_id"], df_source["question_type"])
    )
    out_rows = []

    for exp_name, result in results.items():
        with open(result["cache_paths"]["rag"], "rb") as f:
            rag_runs = pickle.load(f)
        with open(result["cache_paths"]["correctness"], "rb") as f:
            correctness_rows = pickle.load(f)

        corr_map = {
            r["financebench_id"]: int(str(r.get("verdict", "")).lower() == "correct")
            for r in correctness_rows
        }

        doc_name_map = dict(zip(df_source["financebench_id"], df_source["doc_name"]))
        rows = []

        for r in rag_runs:
            hits = page_hits(
                r.get("retrieved_chunks", []),
                expected_pages(r.get("evidence")),
                expected_doc=doc_name_map.get(r["financebench_id"]),
            )
            answer = str(r.get("rag_answer", ""))
            rows.append(
                {
                    "financebench_id": r["financebench_id"],
                    "question_type": r.get("question_type")
                    or question_type_map.get(r["financebench_id"]),
                    "correctness": corr_map.get(r["financebench_id"], 0),
                    "page_hit_at_5": hits.get("page_hit_at_5", 0),
                    "is_refusal": "does not contain the answer" in answer.lower(),
                }
            )

        exp_df = pd.DataFrame(rows)

        row = {
            "experiment": exp_name,
            "n_questions": int(len(exp_df)),
            "refusal_count": int(exp_df["is_refusal"].sum()),
            "refusal_rate": _safe_mean(exp_df["is_refusal"].astype(int)),
            "correctness_given_hit5": _safe_mean(
                exp_df.loc[exp_df["page_hit_at_5"] == 1, "correctness"]
            ),
            "correctness_given_miss5": _safe_mean(
                exp_df.loc[exp_df["page_hit_at_5"] == 0, "correctness"]
            ),
            "domain_correctness": _safe_mean(
                exp_df.loc[exp_df["question_type"] == "domain-relevant", "correctness"]
            ),
            "novel_correctness": _safe_mean(
                exp_df.loc[exp_df["question_type"] == "novel-generated", "correctness"]
            ),
            "domain_page_hit_at_5": _safe_mean(
                exp_df.loc[
                    exp_df["question_type"] == "domain-relevant", "page_hit_at_5"
                ]
            ),
            "novel_page_hit_at_5": _safe_mean(
                exp_df.loc[
                    exp_df["question_type"] == "novel-generated", "page_hit_at_5"
                ]
            ),
        }
        out_rows.append(row)

    return pd.DataFrame(out_rows)


def _metrics_row(result: Dict, metric_cols: list) -> Dict:
    row = {
        "experiment": result["experiment"],
        "change": result["change"],
    }
    row.update({c: result[c] for c in metric_cols})
    return row


def report_experiment_against_baseline(
    exp_name: str,
    exp_result: Dict,
    baseline_result: Optional[Dict] = None,
    df_source: Optional[pd.DataFrame] = None,
) -> Dict[str, pd.DataFrame]:
    """Print a compact baseline comparison right after an experiment finishes."""
    baseline_result = baseline_result or load_baseline_from_task6_caches()
    df_source = df_source if df_source is not None else df
    metric_cols = ["correctness", "faithfulness"] + [f"page_hit_at_{k}" for k in HIT_KS]

    comparison_df = pd.DataFrame(
        [
            _metrics_row(baseline_result, metric_cols),
            _metrics_row(exp_result, metric_cols),
        ]
    )

    delta_df = comparison_df.set_index("experiment")[metric_cols].subtract(
        comparison_df.set_index("experiment").loc["baseline", metric_cols],
        axis="columns",
    )
    delta_df = delta_df.loc[[exp_name]]

    diagnostics_df = build_diagnostics(
        {"baseline": baseline_result, exp_name: exp_result},
        df_source,
    )

    print(f"\n=== Aggregate metrics: baseline vs {exp_name} ===")
    display(comparison_df)
    print(f"\n=== Delta vs baseline: {exp_name} ===")
    display(delta_df.round(3))
    print(f"\n=== Diagnostics: baseline vs {exp_name} ===")
    display(diagnostics_df.round(3))

    return {
        "comparison": comparison_df,
        "delta": delta_df,
        "diagnostics": diagnostics_df,
    }


print("Task 7 harness ready ✓")

Task 7 harness ready ✓


### Experiment 1 - Add reranker (`BAAI/bge-reranker-base`)

**Hypothesis:** Re-ranking top-20 retrieved chunks with a cross-encoder before selecting top-5 for generation should improve retrieval quality (especially page-hit@k), which should increase correctness through better evidence exposure. Faithfulness is expected to stay similar or improve slightly because more evidence-aligned chunks should reach the generator.

**Single change vs baseline:** retrieval only (FAISS top-20 then rerank to top-5). Generation model, prompt, and k=5 at the generator stay unchanged.

In [31]:
from sentence_transformers import CrossEncoder

reranker_model_name = "BAAI/bge-reranker-base"
reranker = CrossEncoder(reranker_model_name)


def rerank_retrieve(query: str, k: int = 5, recall_k: int = 20) -> List[Dict]:
    # Stage 1: dense retrieval for recall.
    candidate_docs = vectorstore.similarity_search(query, k=recall_k)
    if not candidate_docs:
        return []

    candidate_chunks = [_doc_to_chunk_dict(d) for d in candidate_docs]

    # Stage 2: cross-encoder scores each (query, chunk) pair.
    pairs = [(query, c["text"]) for c in candidate_chunks]
    scores = reranker.predict(pairs)

    ranked_idx = sorted(
        range(len(candidate_chunks)),
        key=lambda i: float(scores[i]),
        reverse=True,
    )
    return [candidate_chunks[i] for i in ranked_idx[:k]]


cfg_reranker = PipelineConfig(
    name="reranker",
    change="Add bge-reranker-base: FAISS top-20 -> rerank -> top-5",
    retrieve_fn=rerank_retrieve,
    system_prompt=SYSTEM_PROMPT,
    generation_model=RAG_MODEL,
    k=RAG_K,
)

baseline_result = load_baseline_from_task6_caches()
result_reranker = run_experiment(cfg_reranker)
reranker_report = report_experiment_against_baseline(
    "reranker",
    result_reranker,
    baseline_result=baseline_result,
    df_source=df,
)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: BAAI/bge-reranker-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loaded task7 rag (reranker) (100 rows) from 'task7_reranker_rag_run.pkl'
Loaded task7 correctness (reranker) (100 rows) from 'task7_reranker_correctness.pkl'
Loaded task7 faithfulness (reranker) (20 rows) from 'task7_reranker_faithfulness.pkl'

=== Aggregate metrics: baseline vs reranker ===


,experiment,change,correctness,faithfulness,page_hit_at_1,page_hit_at_3,page_hit_at_5
0,baseline,"Task 6 baseline (k=5, chunk=1000, strict prompt)",0.29,0.423333,0.20,0.33,0.4
1,reranker,Add bge-reranker-base: FAISS top-20 -> rerank ...,0.25,0.288333,0.13,0.22,0.3



=== Delta vs baseline: reranker ===


,correctness,faithfulness,page_hit_at_1,page_hit_at_3,page_hit_at_5
experiment,,,,,
reranker,-0.04,-0.135,-0.07,-0.11,-0.1



=== Diagnostics: baseline vs reranker ===


,experiment,n_questions,refusal_count,refusal_rate,correctness_given_hit5,correctness_given_miss5,domain_correctness,novel_correctness,domain_page_hit_at_5,novel_page_hit_at_5
0,baseline,100,58,0.58,0.55,0.117,0.24,0.34,0.32,0.48
1,reranker,100,61,0.61,0.60,0.100,0.14,0.36,0.20,0.40


**Interpretation:** 

The reranker hurt overall performance relative to the Task 6 baseline: correctness fell from 0.29 to 0.25, faithfulness from 0.423 to 0.288, and page-hit@5 from 0.40 to 0.30 (with page-hit@1 and @3 also dropping). The only positive signal was that correctness conditional on a page-hit@5 improved (see diagnostics above), but because retrieval coverage fell and refusals increased from 58 to 61, the net effect was negative.

### Experiment 2 - Relax generation prompt (reduce hard refusals)

**Hypothesis:** The strict refusal policy in Task 6 likely suppresses correctness when context is partial-but-useful. Allowing grounded partial answers with explicit uncertainty should increase correctness while retrieval metrics remain unchanged; faithfulness may decrease if the model extrapolates beyond strong evidence.

**Single change vs baseline:** generation system prompt only.

In [32]:
RELAXED_SYSTEM_PROMPT = """You are a careful financial-document assistant.

Rules:
- Prioritize answering from the provided CONTEXT only.
- If the context contains supporting evidence, provide the best grounded answer you can.
- If the context is partial, answer only the supported part and explicitly mark uncertainty for unsupported details.
- Refuse only when no relevant evidence is present at all.
- Keep answers concise (1-4 sentences).
- Cite the source document for each factual claim in the form (doc_name, page N).
- Never fabricate citations.
"""

cfg_relaxed_prompt = PipelineConfig(
    name="relaxed_prompt",
    change="Relax system prompt to allow grounded partial answers",
    retrieve_fn=retrieve,
    system_prompt=RELAXED_SYSTEM_PROMPT,
    generation_model=RAG_MODEL,
    k=RAG_K,
)

baseline_result = load_baseline_from_task6_caches()
result_relaxed_prompt = run_experiment(cfg_relaxed_prompt)
relaxed_prompt_report = report_experiment_against_baseline(
    "relaxed_prompt",
    result_relaxed_prompt,
    baseline_result=baseline_result,
    df_source=df,
)

Loaded task7 rag (relaxed_prompt) (100 rows) from 'task7_relaxed_prompt_rag_run.pkl'
Loaded task7 correctness (relaxed_prompt) (100 rows) from 'task7_relaxed_prompt_correctness.pkl'
Loaded task7 faithfulness (relaxed_prompt) (20 rows) from 'task7_relaxed_prompt_faithfulness.pkl'

=== Aggregate metrics: baseline vs relaxed_prompt ===


,experiment,change,correctness,faithfulness,page_hit_at_1,page_hit_at_3,page_hit_at_5
0,baseline,"Task 6 baseline (k=5, chunk=1000, strict prompt)",0.29,0.423333,0.2,0.33,0.4
1,relaxed_prompt,Relax system prompt to allow grounded partial ...,0.33,0.712381,0.2,0.33,0.4



=== Delta vs baseline: relaxed_prompt ===


,correctness,faithfulness,page_hit_at_1,page_hit_at_3,page_hit_at_5
experiment,,,,,
relaxed_prompt,0.04,0.289,0.0,0.0,0.0



=== Diagnostics: baseline vs relaxed_prompt ===


,experiment,n_questions,refusal_count,refusal_rate,correctness_given_hit5,correctness_given_miss5,domain_correctness,novel_correctness,domain_page_hit_at_5,novel_page_hit_at_5
0,baseline,100,58,0.58,0.550,0.117,0.24,0.34,0.32,0.48
1,relaxed_prompt,100,0,0.00,0.575,0.167,0.32,0.34,0.32,0.48


**Interpretation:**

Relaxing the prompt improved correctness from 0.29 to 0.33 (+0.04) and substantially improved faithfulness from 0.423 to 0.712 (+0.289), while retrieval quality stayed unchanged across page-hit metrics. The main behavioral change was eliminating refusals entirely (58% -> 0%), suggesting the strict baseline prompt was suppressing grounded partial answers; gains came mostly from better answering when retrieval missed, with domain correctness improving from 0.24 to 0.32.

### Experiment 3 - Change chunk size to 500

**Hypothesis:** Smaller chunks (`chunk_size=500`, `chunk_overlap=75`) should improve evidence localization for long table-heavy pages and increase page-hit@k. Correctness may improve via better retrieval, while faithfulness may move either direction due to reduced context per chunk.

**Single change vs baseline:** chunking policy + FAISS index (new index folder), everything else unchanged.

In [33]:
VECTORSTORE_DIR_CHUNK500 = "financebench_rag_faiss_chunk500"

if os.path.isdir(VECTORSTORE_DIR_CHUNK500):
    vectorstore_chunk500 = FAISS.load_local(
        VECTORSTORE_DIR_CHUNK500,
        embeddings,
        allow_dangerous_deserialization=True,
    )
    print(
        f"Loaded chunk500 FAISS index from {VECTORSTORE_DIR_CHUNK500!r} "
        f"({vectorstore_chunk500.index.ntotal} vectors)"
    )
else:
    all_pages_chunk500 = []
    for _, row in doc_rows.iterrows():
        loader = PyPDFLoader(row["doc_link"])
        pages = loader.load()

        for page_number, doc in enumerate(pages):
            doc.metadata = doc.metadata or {}
            doc.metadata["doc_name"] = row["doc_name"]
            doc.metadata["company"] = row["company"]
            doc.metadata["doc_period"] = row["doc_period"]
            doc.metadata["page_number"] = page_number
            all_pages_chunk500.append(doc)

    splitter_chunk500 = RecursiveCharacterTextSplitter(
        chunk_size=500,
        chunk_overlap=75,
    )
    chunks_chunk500 = splitter_chunk500.split_documents(all_pages_chunk500)
    vectorstore_chunk500 = FAISS.from_documents(chunks_chunk500, embeddings)
    vectorstore_chunk500.save_local(VECTORSTORE_DIR_CHUNK500)
    print(
        f"Built and saved chunk500 FAISS index to {VECTORSTORE_DIR_CHUNK500!r} "
        f"({vectorstore_chunk500.index.ntotal} vectors)"
    )

Loaded chunk500 FAISS index from 'financebench_rag_faiss_chunk500' (46736 vectors)


In [34]:
def retrieve_chunk500(query: str, k: int = 5) -> List[Dict]:
    docs = vectorstore_chunk500.similarity_search(query, k=k)
    return [_doc_to_chunk_dict(d) for d in docs]


cfg_chunk500 = PipelineConfig(
    name="chunk500",
    change="Rebuild index with chunk_size=500, chunk_overlap=75",
    retrieve_fn=retrieve_chunk500,
    system_prompt=SYSTEM_PROMPT,
    generation_model=RAG_MODEL,
    k=RAG_K,
)

baseline_result = load_baseline_from_task6_caches()
result_chunk500 = run_experiment(cfg_chunk500)
chunk500_report = report_experiment_against_baseline(
    "chunk500",
    result_chunk500,
    baseline_result=baseline_result,
    df_source=df,
)

Loaded task7 rag (chunk500) (100 rows) from 'task7_chunk500_rag_run.pkl'
Loaded task7 correctness (chunk500) (100 rows) from 'task7_chunk500_correctness.pkl'
Loaded task7 faithfulness (chunk500) (20 rows) from 'task7_chunk500_faithfulness.pkl'

=== Aggregate metrics: baseline vs chunk500 ===


,experiment,change,correctness,faithfulness,page_hit_at_1,page_hit_at_3,page_hit_at_5
0,baseline,"Task 6 baseline (k=5, chunk=1000, strict prompt)",0.29,0.423333,0.20,0.33,0.40
1,chunk500,"Rebuild index with chunk_size=500, chunk_overl...",0.24,0.318750,0.17,0.30,0.36



=== Delta vs baseline: chunk500 ===


,correctness,faithfulness,page_hit_at_1,page_hit_at_3,page_hit_at_5
experiment,,,,,
chunk500,-0.05,-0.105,-0.03,-0.03,-0.04



=== Diagnostics: baseline vs chunk500 ===


,experiment,n_questions,refusal_count,refusal_rate,correctness_given_hit5,correctness_given_miss5,domain_correctness,novel_correctness,domain_page_hit_at_5,novel_page_hit_at_5
0,baseline,100,58,0.58,0.550,0.117,0.24,0.34,0.32,0.48
1,chunk500,100,66,0.66,0.528,0.078,0.20,0.28,0.24,0.48


**Interpretation:**

Reducing the chunk size to 500 hurt performance across the board: correctness dropped from 0.29 to 0.24 (-0.05), faithfulness dropped from 0.423 to 0.319 (-0.105), and retrieval quality also declined at every cutoff, especially page_hit@5 (0.40 -> 0.36). The diagnostics show that correctness conditional on a page-hit stayed similar while correctness when retrieval missed fell further, suggesting the main issue is worse retrieval coverage rather than answer generation quality when evidence is retrieved.

### Aggregate comparison tables + xlsx export

The cells below compute all Task 7 comparison numbers in code (no narrative conclusions).

In [35]:
# Final table: combine every experiment result that has already been run.
baseline_result = load_baseline_from_task6_caches()

completed_results = {"baseline": baseline_result}
optional_results = {
    "reranker": "result_reranker",
    "relaxed_prompt": "result_relaxed_prompt",
    "chunk500": "result_chunk500",
}

for exp_name, var_name in optional_results.items():
    if var_name in globals():
        completed_results[exp_name] = globals()[var_name]
    else:
        print(f"Skipping {exp_name!r}: run its experiment cell first.")

order = ["baseline", "reranker", "relaxed_prompt", "chunk500"]
metric_cols = ["correctness", "faithfulness"] + [f"page_hit_at_{k}" for k in HIT_KS]

improvement_df = pd.DataFrame(
    [
        _metrics_row(completed_results[name], metric_cols)
        for name in order
        if name in completed_results
    ]
)

TASK_7_FILENAME = "assignment2_improvement_cycles.xlsx"
improvement_df.to_excel(TASK_7_FILENAME, index=False)
print(f"Saved {TASK_7_FILENAME} with {len(improvement_df)} rows")
print("\nTask 7 aggregate table:")
display(improvement_df)

if len(improvement_df) < len(order):
    print("\nNote: final submission should include baseline + all three experiments.")

Saved assignment2_improvement_cycles.xlsx with 4 rows

Task 7 aggregate table:


,experiment,change,correctness,faithfulness,page_hit_at_1,page_hit_at_3,page_hit_at_5
0,baseline,"Task 6 baseline (k=5, chunk=1000, strict prompt)",0.29,0.423333,0.20,0.33,0.40
1,reranker,Add bge-reranker-base: FAISS top-20 -> rerank ...,0.25,0.288333,0.13,0.22,0.30
2,relaxed_prompt,Relax system prompt to allow grounded partial ...,0.33,0.712381,0.20,0.33,0.40
3,chunk500,"Rebuild index with chunk_size=500, chunk_overl...",0.24,0.318750,0.17,0.30,0.36


In [36]:
# Final comparisons across all completed experiments.
baseline_metrics = improvement_df.set_index("experiment").loc["baseline", metric_cols]
delta_df = improvement_df.set_index("experiment")[metric_cols].subtract(
    baseline_metrics,
    axis="columns",
)
delta_df = delta_df.drop(index="baseline")

print("Delta vs baseline:")
display(delta_df.round(3))

# Diagnostic breakdown by experiment.
# Includes refusal counts, conditional correctness by page_hit@5, and by-question_type splits.
diagnostics_df = build_diagnostics(completed_results, df)
print("\nDiagnostics by experiment:")
display(diagnostics_df.round(3))

Delta vs baseline:


,correctness,faithfulness,page_hit_at_1,page_hit_at_3,page_hit_at_5
experiment,,,,,
reranker,-0.04,-0.135,-0.07,-0.11,-0.10
relaxed_prompt,0.04,0.289,0.00,0.00,0.00
chunk500,-0.05,-0.105,-0.03,-0.03,-0.04



Diagnostics by experiment:


,experiment,n_questions,refusal_count,refusal_rate,correctness_given_hit5,correctness_given_miss5,domain_correctness,novel_correctness,domain_page_hit_at_5,novel_page_hit_at_5
0,baseline,100,58,0.58,0.550,0.117,0.24,0.34,0.32,0.48
1,reranker,100,61,0.61,0.600,0.100,0.14,0.36,0.20,0.40
2,relaxed_prompt,100,0,0.00,0.575,0.167,0.32,0.34,0.32,0.48
3,chunk500,100,66,0.66,0.528,0.078,0.20,0.28,0.24,0.48


### Task 7 - wrap-up

> 1. Where does the pipeline fail most (retrieval, generation, or both)?
> 2. Which experiment performed best, and by which metrics?
> 3. If you had one more week, what would you try next and why?


The pipeline fails in both retrieval and generation, but the largest structural bottleneck is retrieval: even the baseline retrieves the evidence page in the top 5 for only 40% of questions, so many answers never receive the right context. Generation also contributes because the strict baseline prompt refuses 58% of questions; relaxing the prompt removes refusals and improves correctness even though retrieval metrics stay unchanged.

The best experiment was `relaxed_prompt`: it achieved the highest correctness (0.33 vs. 0.29 baseline) and the highest faithfulness (0.712 vs. 0.423), with no loss in page-hit metrics. By contrast, both the reranker and smaller chunks degraded retrieval quality and reduced correctness, suggesting these retrieval changes were not well matched to the dataset or query style.

With one more week, I would focus on retrieval rather than chunk size alone: try query rewriting / multi-query retrieval, metadata-aware filtering by company and document period, and a reranker tuned or evaluated on finance-style evidence matching. I would keep the relaxed prompt, since it improved answer behavior without changing retrieval.